# Datos para el mapa web

## Objetivo
Preparar los datos del mapa interactivo `mapa_paises.html`, que es una versión más completa de la
pregunta 1. Además del selector de feature, tiene:
- filtros por **género** y por **franja de edad** de los usuarios
- un control de **año** (2005 a 2014),
- un panel por país con su perfil, los **artistas** más escuchados y una **comparación** con otro país,
- un **buscador de artista** que pinta el mapa según el porcentaje de usuarios de cada país que lo escucha.

Como el mapa corre en el navegador, no puede consultar DuckDB. Lo que hacemos acá es dejar las tablas calculadas (etapa de **Transformación**) y guardarlas en archivos que el mapa lee.

- Este notebook necesita que ya se haya corrido `03_tablas.ipynb`.

## 1. Preparación del entorno

In [25]:
import json
import urllib.request
from pathlib import Path
import pandas as pd
import pycountry

from config import q, guardar, ruta_tablas, carpeta, FEATURES, MIN_USUARIOS_PAIS

# Carpeta donde van los archivos del mapa (ahí también tiene que estar mapa_paises.html)
carpeta_web = carpeta / "mapa_web"
carpeta_web.mkdir(exist_ok=True)

# Valores que elegimos para el mapa
MIN_ESCUCHAS_ANIO = 30      # un usuario cuenta en un año si tiene al menos 30 escuchas con datos de audio ese año
N_ARTISTAS_MAPA = 5000      # cantidad de artistas con más oyentes que entran al buscador

print("Carpeta del mapa:", carpeta_web)

Carpeta del mapa: C:\Users\mgrau\musica-por-pais\mapa_web


## 2. Tabla por país, género y franja de edad

Para que los filtros funcionen en el navegador, guardamos una fila por **país, género y franja
de edad**, con la cantidad de usuarios y la **suma** de cada feature (no el promedio).

Se guardan sumas porque el promedio de varios grupos no es el promedio de sus promedios: un grupo
de 200 usuarios pesa más que uno de 5. Con la suma y la cantidad de usuarios, el mapa puede juntar
los grupos que elija el usuario (por ejemplo, mujeres de 20 a 24 años) y recién ahí dividir.

Los usuarios sin edad válida tienen la franja nula; acá los llamamos `sin edad`.

In [39]:
# Generamos una suma para cada feature incluida en FEATURES
sumas = ",\n           ".join(f"SUM({f}) AS {f}" for f in FEATURES)

# Armamos una tabla agrupada por país, género y franja de edad
# para poder aplicar los filtros del mapa.
guardar(f"""
    SELECT country,
           gender,
           -- Agrupamos a los usuarios sin edad válida como 'sin edad'
           COALESCE(franja_edad, 'sin edad') AS franja_edad,

           -- Contamos los usuarios de cada grupo
           COUNT(*) AS n_usuarios,

           -- Sumamos los valores de cada feature
           {sumas}
    FROM tabla_usuario

    -- Nos quedamos con los países que cumplen con el mínimo de usuarios
    WHERE country IN (SELECT country FROM tabla_usuario
                      GROUP BY country HAVING COUNT(*) >= {MIN_USUARIOS_PAIS})

    -- Agrupamos por país, género y franja de edad
    GROUP BY country, gender, COALESCE(franja_edad, 'sin edad')
""", "pais_segmento")

pais_segmento: 783 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/pais_segmento.parquet


In [41]:
#Los usuarios de todos los grupos tienen que dar el total de los países que quedaron
# Cargamos la tabla pais_segmento
seg = q(f"SELECT * FROM '{ruta_tablas}pais_segmento.parquet'")

# Revisamos la cantidad de filas, países y usuarios
print("Filas:", len(seg), "| Países:", seg["country"].nunique())
print("Usuarios en total:", f"{seg['n_usuarios'].sum():,}")

# Revisamos los géneros y las franjas de edad que quedaron
print("Géneros:", sorted(seg["gender"].unique()))
print("Franjas:", sorted(seg["franja_edad"].unique()))

# Contamos los usuarios de tabla_usuario aplicando el mismo filtro de países
total = q(f"""
    SELECT COUNT(*) AS n FROM tabla_usuario
    WHERE country IN (SELECT country FROM tabla_usuario
                      GROUP BY country HAVING COUNT(*) >= {MIN_USUARIOS_PAIS})
""")["n"][0]

print("Usuarios de esos países en tabla_usuario:", f"{total:,}")

Filas: 783 | Países: 56
Usuarios en total: 52,098
Géneros: ['f', 'm', 'no informado']
Franjas: ['10-19', '20-24', '25-29', '30+', 'sin edad']
Usuarios de esos países en tabla_usuario: 52,098


## 3. Tabla por año

Para el control de año necesitamos el promedio de cada usuario **en cada año**. Es el mismo cálculo
que `tabla_usuario`, pero agrupando por usuario y año. 
Calculamos el promedio de cada feature por usuario y año para poder analizar cómo cambian los gustos musicales a lo largo del tiempo. Al calcular primero los promedios individuales, evitamos que los usuarios con más escuchas tengan mayor peso en el perfil musical de cada país. Mantenemos así el mismo criterio que utilizamos en tabla_usuario.

**Filtrado:** un usuario cuenta en un año solo si ese año tiene al menos `MIN_ESCUCHAS_ANIO` escuchas
con datos de audio. Con menos, el promedio de ese usuario en ese año depende de muy pocas canciones.
Usamos solo los usuarios de `tabla_usuario`, así los países y los grupos son los mismos que en el resto.

Hay que tener en cuenta que 2005 empieza en febrero, 2014 termina en septiembre y en los primeros
años hay menos usuarios activos.

In [42]:
# Obtenemos los valores mínimos y máximos de tempo y loudness
# para normalizarlos con la misma escala que usamos en tabla_usuario
r = q("""
    SELECT MIN(tempo) AS tempo_min, MAX(tempo) AS tempo_max,
           MIN(loudness) AS loud_min, MAX(loudness) AS loud_max
    FROM acoustic_features_clean
""").iloc[0]

tempo_min, tempo_max, loud_min, loud_max = r["tempo_min"], r["tempo_max"], r["loud_min"], r["loud_max"]

# Armamos una tabla con una fila por usuario y año
# para analizar cómo cambian los gustos musicales a lo largo del tiempo
guardar(f"""
    SELECT e.user_id,

           -- Obtenemos el año de cada escucha
           EXTRACT(year FROM to_timestamp(e."timestamp"))::INTEGER AS anio,

           -- Contamos las escuchas de cada usuario en el año
           COUNT(*)                 AS n_escuchas,

           -- Calculamos el promedio de cada característica musical
           AVG(f.danceability)      AS danceability,
           AVG(f.energy)            AS energy,
           AVG(f.valence)           AS valence,
           AVG(f.acousticness)      AS acousticness,
           AVG(f.instrumentalness)  AS instrumentalness,
           AVG(f.speechiness)       AS speechiness,
           AVG(f.liveness)          AS liveness,

           -- Normalizamos tempo y loudness entre 0 y 1 y calculamos sus promedios
           AVG((f.tempo - {tempo_min}) / ({tempo_max} - {tempo_min})) AS tempo_norm,
           AVG((f.loudness - {loud_min}) / ({loud_max} - {loud_min})) AS loudness_norm

    FROM events_clean e

    -- Unimos las escuchas con las características musicales de cada canción
    JOIN acoustic_features_clean f ON e.track_id = f.track_id

    -- Conservamos los usuarios incluidos en tabla_usuario
    WHERE e.user_id IN (SELECT user_id FROM tabla_usuario)

    -- Agrupamos las escuchas por usuario y año
    GROUP BY e.user_id, anio

    -- Conservamos las combinaciones de usuario y año
    -- que cumplen con el mínimo de escuchas definido
    HAVING COUNT(*) >= {MIN_ESCUCHAS_ANIO}
""", "usuario_anio")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

usuario_anio: 194,539 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/usuario_anio.parquet


Revisamos cuántos usuarios activos hay en cada año. Si en algún año hay muy pocos, los países van a
quedar en gris en el mapa para ese año.

In [43]:
q(f"""
    SELECT anio, COUNT(*) AS usuarios, SUM(n_escuchas) AS escuchas
    FROM '{ruta_tablas}usuario_anio.parquet'
    GROUP BY anio ORDER BY anio
""")

,anio,usuarios,escuchas
0,2005,534,786035.0
1,2006,1669,2868317.0
2,2007,3789,7102274.0
3,2008,7085,12648273.0
4,2009,11803,21634777.0
5,2010,18958,34489368.0
6,2011,30869,55013458.0
7,2012,48282,90398046.0
8,2013,40858,91576525.0
9,2014,30692,33968014.0


Juntamos los usuarios por **país, año, género y franja de edad** y guardamos la
cantidad de usuarios y la suma de cada feature.

In [45]:
# Generamos la suma de cada feature para los datos anuales
sumas_anio = ",\n           ".join(f"SUM(a.{f}) AS {f}" for f in FEATURES)

# Armamos una tabla agrupada por país, año, género y franja de edad
# para poder combinar los filtros del mapa interactivo
guardar(f"""
    SELECT u.country,
           a.anio,
           u.gender,

           -- Agrupamos a los usuarios sin edad válida como 'sin edad'
           COALESCE(u.franja_edad, 'sin edad') AS franja_edad,

           -- Contamos los usuarios y sumamos los valores de cada feature
           COUNT(*) AS n_usuarios,
           {sumas_anio}

    FROM '{ruta_tablas}usuario_anio.parquet' a

    -- Incorporamos el país, género y franja de edad de cada usuario
    JOIN tabla_usuario u ON a.user_id = u.user_id

    -- Conservamos los países que cumplen con el mínimo de usuarios
    WHERE u.country IN (SELECT country FROM tabla_usuario
                        GROUP BY country HAVING COUNT(*) >= {MIN_USUARIOS_PAIS})

    -- Agrupamos según los filtros disponibles en el mapa
    GROUP BY u.country, a.anio, u.gender, COALESCE(u.franja_edad, 'sin edad')
""", "pais_segmento_anio")

# Cargamos la tabla generada
seg_anio = q(f"SELECT * FROM '{ruta_tablas}pais_segmento_anio.parquet'")

# Verificamos la cantidad de filas y los años disponibles
print("Filas:", len(seg_anio), "| Años:", sorted(seg_anio["anio"].unique()))

pais_segmento_anio: 5,707 filas guardadas en C:/Users/mgrau/musica-por-pais/data/tablas/pais_segmento_anio.parquet
Filas: 5707 | Años: [np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014)]


## 4. Artistas

Usamos la tabla `pais_artista` que se arma en `03_tablas.ipynb` (una fila por país y artista, con los
**oyentes**, o sea usuarios distintos que escucharon al artista, y `pct_pais`, el porcentaje de los
usuarios del país que lo escucha). Contamos oyentes y no escuchas para que un solo usuario que pone la
misma banda miles de veces no decida el resultado.

Con esa tabla armamos dos cosas para el mapa:
- **Más escuchados:** los 5 artistas con mayor `pct_pais` en cada país, para el panel del país.
- **Buscador:** los `N_ARTISTAS_MAPA` artistas con más oyentes en total, cada uno con el porcentaje en
  cada país. Con eso el mapa puede pintarse según un artista.

In [31]:
archivo = Path(ruta_tablas) / "pais_artista.parquet"
assert archivo.exists(), "Falta pais_artista.parquet: correr antes 03_tablas.ipynb"

pais_artista = q(f"SELECT country, artist_id, artist_name, oyentes, pct_pais FROM '{archivo.as_posix()}'")
print("Filas:", f"{len(pais_artista):,}", "| Países:", pais_artista["country"].nunique(),
      "| Artistas:", pais_artista["artist_id"].nunique())

# Más escuchados por país: mayor porcentaje de usuarios del país
mas_escuchados = (pais_artista.sort_values(["country", "pct_pais"], ascending=[True, False])
                  .groupby("country").head(5))

# Ejemplo: Argentina
mas_escuchados[mas_escuchados["country"] == "AR"][["artist_name", "oyentes", "pct_pais"]]

Filas: 458,744 | Países: 56 | Artistas: 10000


,artist_name,oyentes,pct_pais
108010,Radiohead,138,0.514925
91425,Daft Punk,120,0.447761
100657,The Strokes,119,0.444030
195826,Arctic Monkeys,117,0.436567
114167,The Cure,108,0.402985


In [32]:
# Los artistas con más oyentes en total (sumando todos los países)
total_artista = (pais_artista.groupby(["artist_id", "artist_name"])["oyentes"].sum()
                 .sort_values(ascending=False).head(N_ARTISTAS_MAPA))
print("Artistas en el buscador:", len(total_artista))
print("Los 5 primeros:", [nombre for (_, nombre) in total_artista.index[:5]])

# Solo las filas de esos artistas, agrupadas por artista para armar la lista rápido
ids_buscador = set(i for (i, _) in total_artista.index)
por_artista = {k: g for k, g in pais_artista[pais_artista["artist_id"].isin(ids_buscador)].groupby("artist_id")}

paises_tabla = set(seg_anio["country"])
print("Países sin artistas más escuchados:", sorted(paises_tabla - set(mas_escuchados["country"])))

Artistas en el buscador: 5000
Los 5 primeros: ['Daft Punk', 'Radiohead', 'Coldplay', 'Foo Fighters', 'Arctic Monkeys']
Países sin artistas más escuchados: []


## 5. Formas de los países

El mapa necesita las formas (polígonos) de los países. Usamos las de **Natural Earth**, que son de
dominio público, en el archivo `ne_50m_admin_0_countries.geojson` (escala 1:50 millones, un buen
equilibrio entre detalle y tamaño). Se descarga una sola vez.

De ese archivo nos quedamos con el código de 2 letras y el nombre de cada país. Natural Earth deja
`-99` en el código de algunos países (por ejemplo Francia y Noruega) y para esos usa otra columna,
`ISO_A2_EH`, que sí lo tiene. También redondeamos las coordenadas a 2 decimales para que el archivo
pese menos, y sacamos la Antártida, que no tiene datos y ocupa lugar en el mapa.

In [33]:
URL_PAISES = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson"
archivo_geo = carpeta_web / "ne_50m_admin_0_countries.geojson"

if not archivo_geo.exists():
    try:
        urllib.request.urlretrieve(URL_PAISES, archivo_geo)
    except Exception as error:
        raise RuntimeError(f"No se pudo descargar el archivo ({error}). Bajalo a mano desde {URL_PAISES} "
                           f"y guardalo en {archivo_geo}")

with open(archivo_geo, encoding="utf-8") as f:
    geo = json.load(f)
print("Países en el archivo:", len(geo["features"]))
print("Columnas de cada país:", list(geo["features"][0]["properties"].keys())[:12], "...")

Países en el archivo: 242
Columnas de cada país: ['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF'] ...


In [34]:
def codigo_iso2(propiedades):
    # Usa ISO_A2_EH cuando ISO_A2 viene como -99
    for clave in ["ISO_A2_EH", "ISO_A2"]:
        valor = propiedades.get(clave)
        if valor and valor != "-99":
            return valor
    return None

def redondear(coordenadas):
    # Redondea a 2 decimales una lista de coordenadas, tenga la profundidad que tenga
    if isinstance(coordenadas[0], (int, float)):
        return [round(coordenadas[0], 2), round(coordenadas[1], 2)]
    return [redondear(c) for c in coordenadas]

formas = []
for pais in geo["features"]:
    codigo = codigo_iso2(pais["properties"])
    if codigo is None or codigo == "AQ":     # sin código, o la Antártida
        continue
    formas.append({
        "type": "Feature",
        "properties": {"iso2": codigo, "nombre": pais["properties"].get("NAME", codigo)},
        "geometry": {"type": pais["geometry"]["type"], "coordinates": redondear(pais["geometry"]["coordinates"])},
    })

paises_geo = {"type": "FeatureCollection", "features": formas}
print("Países que quedaron:", len(formas))

# Países de nuestra tabla que no tienen forma en el mapa (no se van a poder pintar)
codigos_mapa = {f["properties"]["iso2"] for f in formas}
print("Países sin forma en el mapa:", sorted(paises_tabla - codigos_mapa))

Países que quedaron: 238
Países sin forma en el mapa: ['AQ']


In [35]:
destino_geo = carpeta_web / "paises_geo.js"
# Se guarda como archivo .js (const PAISES = ...) para que el HTML lo pueda cargar sin servidor
texto = "const PAISES = " + json.dumps(paises_geo, separators=(",", ":")) + ";"
destino_geo.write_text(texto, encoding="utf-8")
print(f"{destino_geo.name}: {destino_geo.stat().st_size / 1e6:.1f} MB")

paises_geo.js: 1.4 MB


## 6. Exportar los datos para el mapa

Guardamos dos archivos para el mapa:
- `datos_mapa.js`: `features`, `generos`, `franjas` y `anios` (las listas en el orden en que se usan los
  números) y, para cada país, el nombre, las filas de todos los años juntos (`filas`), las filas por año
  (`filas_anio`) y sus 5 artistas más escuchados.
- `artistas_mapa.js`: la lista del buscador, con el porcentaje de usuarios de cada país que escucha a cada artista.

En las filas, el género, la franja y el año se guardan como números (la posición en las listas) para
que el archivo pese menos.

In [36]:
generos = ["m", "f", "no informado"]
franjas = ["10-19", "20-24", "25-29", "30+", "sin edad"]
anios = sorted(int(a) for a in seg_anio["anio"].unique())

seg["g"] = seg["gender"].map({g: i for i, g in enumerate(generos)})
seg["fr"] = seg["franja_edad"].map({f: i for i, f in enumerate(franjas)})
seg_anio["g"] = seg_anio["gender"].map({g: i for i, g in enumerate(generos)})
seg_anio["fr"] = seg_anio["franja_edad"].map({f: i for i, f in enumerate(franjas)})
seg_anio["ai"] = seg_anio["anio"].map({a: i for i, a in enumerate(anios)})
assert seg["g"].notna().all() and seg["fr"].notna().all(), "Hay géneros o franjas que no están en las listas"
assert seg_anio["g"].notna().all() and seg_anio["fr"].notna().all(), "Hay géneros o franjas que no están en las listas"

def nombre_pais(codigo):
    # Nombre del país en inglés; si pycountry no lo conoce, queda el código
    pais = pycountry.countries.get(alpha_2=codigo)
    return getattr(pais, "common_name", None) or (pais.name if pais else codigo)

def filas_de(tabla, columnas):
    # Arma una lista de listas con las columnas pedidas y la suma de cada feature, redondeada
    return [[int(fila[c]) for c in columnas] + [round(float(fila[f]), 4) for f in FEATURES]
            for _, fila in tabla.iterrows()]

datos_paises = {}
for codigo, grupo in seg.groupby("country"):
    datos_paises[codigo] = {
        "nombre": nombre_pais(codigo),
        "filas": filas_de(grupo, ["g", "fr", "n_usuarios"]),
        "filas_anio": [],
        "oyentes": [],
    }

for codigo, grupo in seg_anio.groupby("country"):
    datos_paises[codigo]["filas_anio"] = filas_de(grupo, ["ai", "g", "fr", "n_usuarios"])

for fila in mas_escuchados.itertuples():
    datos_paises[fila.country]["oyentes"].append([fila.artist_name, round(float(fila.pct_pais), 3)])

datos_mapa = {"features": FEATURES, "generos": generos, "franjas": franjas, "anios": anios,
              "min_usuarios_pais": MIN_USUARIOS_PAIS, "min_escuchas_anio": MIN_ESCUCHAS_ANIO,
              "paises": datos_paises}

destino_datos = carpeta_web / "datos_mapa.js"
texto = "const DATOS = " + json.dumps(datos_mapa, ensure_ascii=False, separators=(",", ":")) + ";"
destino_datos.write_text(texto, encoding="utf-8")
print(f"{destino_datos.name}: {destino_datos.stat().st_size / 1e6:.2f} MB, {len(datos_paises)} países")

datos_mapa.js: 0.49 MB, 56 países


In [37]:
# Lista del buscador: [nombre, oyentes en total, [[país, % de usuarios del país, oyentes], ...]]
artistas = []
for (artist_id, nombre), total in total_artista.items():
    g = por_artista[artist_id]
    paises_del_artista = [[c, round(float(p), 3), int(o)] for c, p, o in zip(g["country"], g["pct_pais"], g["oyentes"])]
    artistas.append([nombre, int(total), paises_del_artista])

destino_artistas = carpeta_web / "artistas_mapa.js"
texto = "const ARTISTAS = " + json.dumps(artistas, ensure_ascii=False, separators=(",", ":")) + ";"
destino_artistas.write_text(texto, encoding="utf-8")
print(f"{destino_artistas.name}: {destino_artistas.stat().st_size / 1e6:.2f} MB, {len(artistas)} artistas")

artistas_mapa.js: 4.06 MB, 5000 artistas


Una última revisión: un país y un artista de ejemplo, para ver que se guardó lo que esperamos.

In [38]:
ar = datos_paises["AR"]
print(ar["nombre"], "| grupos (todos los años):", len(ar["filas"]), "| grupos (por año):", len(ar["filas_anio"]))
print("Primera fila [género, franja, usuarios, suma de cada feature]:", ar["filas"][0])
print("Primera fila por año [año, género, franja, usuarios, sumas]:", ar["filas_anio"][0])
print("Más escuchados:", ar["oyentes"])
print("Primer artista del buscador:", artistas[0][0], "| países:", len(artistas[0][2]))

Argentina | grupos (todos los años): 15 | grupos (por año): 101
Primera fila [género, franja, usuarios, suma de cada feature]: [0, 0, 29, 14.4229, 20.553, 12.897, 5.4957, 5.6517, 2.1326, 5.9038, 12.4514, 22.7941]
Primera fila por año [año, género, franja, usuarios, sumas]: [6, 0, 4, 14, 6.8126, 9.4611, 6.3085, 3.1225, 2.5819, 0.8548, 2.7616, 5.9447, 10.8565]
Más escuchados: [['Radiohead', 0.515], ['Daft Punk', 0.448], ['The Strokes', 0.444], ['Arctic Monkeys', 0.437], ['The Cure', 0.403]]
Primer artista del buscador: Daft Punk | países: 56


## 7. Cómo abrir el mapa

1. Copiar `mapa_paises.html` a la carpeta `mapa_web/` (la misma donde quedaron `datos_mapa.js`, `paises_geo.js` y `artistas_mapa.js`).
2. Abrirlo con doble click en el navegador. Hace falta conexión a internet, porque el mapa de fondo y la
   librería MapLibre se bajan de internet.
3. Si el navegador no carga los datos, abrir una terminal en esa carpeta, correr `python -m http.server`
   y entrar a `http://localhost:8000/mapa_paises.html`.

Estos archivos se pueden volver a generar corriendo este notebook, así que no hace falta subirlos a git
(los parquet tampoco se suben).